# Week 10 -- Cost-Aware Fraud Decision Optimization

**FraudFlow Project | Week 10**

Optimizing operational fraud-detection decisions using a cost-aware threshold. We explicitly balance the cost of false negatives (fraud loss) against the cost of false positives (manual review/customer friction).


## 1. Objective

- Define a cost framework incorporating FN loss and FP operational cost.
- Compare existing predictive models (Logistic Regression, Decision Tree, Random Forest, XGBoost).
- Find the optimal operational threshold that minimizes Total Expected Monetary Cost.
- Assess probability calibration.
- Prevent data leakage (threshold tuned on validation, not test).


## 2. Load Data and Reproduce Splits


In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_recall_curve, brier_score_loss

from src.preprocessing.feature_engineering import engineer_features
from scripts.run_week10 import calculate_expected_cost, FP_COST, RANDOM_STATE

print('Loading data...')
df_raw = pd.read_csv('data/raw/fraudTest.csv')
df = engineer_features(df_raw)

loss_amount = df['fraud_loss_amount'] if 'fraud_loss_amount' in df.columns else df['amt']
y = df['is_fraud']
X = df.drop(columns=['is_fraud', 'fraud_loss_amount'], errors='ignore')

X_train_full, X_test, y_train_full, y_test, loss_train_full, loss_test = train_test_split(
    X, y, loss_amount, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val, loss_train, loss_val = train_test_split(
    X_train_full, y_train_full, loss_train_full, test_size=0.2, stratify=y_train_full, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## 3. Load Existing Models

Reusing models trained in previous weeks. Preprocessor from Week 8 is used.


In [ ]:
MODELS_DIR = Path('models')
xgb_art = joblib.load(MODELS_DIR / 'xgboost_week8.joblib')
preprocessor = xgb_art['preprocessor']

models = {
    'XGBoost': xgb_art['model'],
    'Logistic Regression': joblib.load(MODELS_DIR / 'logistic_regression_balanced_week4.joblib'),
    'Decision Tree': joblib.load(MODELS_DIR / 'decision_tree_week6.joblib'),
}
rf_art = joblib.load(MODELS_DIR / 'random_forest_week7.joblib')
models['Random Forest'] = rf_art['model'] if isinstance(rf_art, dict) and 'model' in rf_art else rf_art

X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

if hasattr(X_val_proc, 'toarray'): X_val_proc = X_val_proc.toarray().astype('float32')
if hasattr(X_test_proc, 'toarray'): X_test_proc = X_test_proc.toarray().astype('float32')
print('Models loaded successfully.')

## 4. Cost Framework Definition

- **False Negative (FN) Cost**: The monetary value of the missed fraudulent transaction (`fraud_loss_amount` or `amt`).
- **False Positive (FP) Cost**: A configured operational cost for reviewing/declining a legitimate transaction.
- **Total Expected Cost**: `(FP * FP_COST) + Sum(FN Loss Amounts)`


In [ ]:
print(f'Configured Operational Cost per False Positive: ${FP_COST:.2f}')

## 5. Threshold Optimization & Model Comparison (Validation Set)


In [ ]:
thresholds = np.arange(0.01, 1.00, 0.01)
comparison = []

for name, model in models.items():
    y_prob = model.predict_proba(X_val_proc)[:, 1]
    best_cost = float('inf')
    best_t = 0.5
    
    for t in thresholds:
        metrics = calculate_expected_cost(y_val, y_prob, loss_val, t, FP_COST)
        if metrics['expected_cost'] < best_cost:
            best_cost = metrics['expected_cost']
            best_t = t
            
    pr_auc = average_precision_score(y_val, y_prob)
    brier = brier_score_loss(y_val, y_prob)
    comparison.append({
        'Model': name,
        'PR-AUC': pr_auc,
        'Brier': brier,
        'Optimal Threshold': best_t,
        'Expected Cost': best_cost
    })

df_comp = pd.DataFrame(comparison).sort_values('Expected Cost')
display(df_comp)

## 6. Calibration Analysis

Models with low Brier scores are better calibrated (probabilities closely match actual risk).


In [ ]:
print('Brier Scores (lower is better):')
for _, row in df_comp.iterrows():
    print(f"{row['Model']}: {row['Brier']:.4f}")
print('\nXGBoost typically benefits from isotonic calibration if extreme precision is required, '
      'but we will use its optimal threshold directly to preserve exact outputs.')

## 7. Expected Cost vs Threshold Visualization


In [ ]:
best_model_name = df_comp.iloc[0]['Model']
best_model = models[best_model_name]
print(f'Visualizing threshold curve for {best_model_name}')

y_prob = best_model.predict_proba(X_val_proc)[:, 1]
costs = [calculate_expected_cost(y_val, y_prob, loss_val, t, FP_COST)['expected_cost'] for t in thresholds]

plt.figure(figsize=(8, 5))
plt.plot(thresholds, costs, color='red', lw=2, label='Total Cost')
plt.xlabel('Decision Threshold')
plt.ylabel('Cost ($)')
plt.title(f'Expected Cost vs Threshold - {best_model_name}')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 8. Final Test Evaluation

Evaluating the best model at its optimal threshold on the UNTOUCHED test set.


In [ ]:
best_threshold = df_comp.iloc[0]['Optimal Threshold']
y_prob_test = best_model.predict_proba(X_test_proc)[:, 1]
test_metrics = calculate_expected_cost(y_test, y_prob_test, loss_test, best_threshold, FP_COST)
test_pr_auc = average_precision_score(y_test, y_prob_test)

print(f'--- FINAL TEST METRICS ({best_model_name} @ T={best_threshold}) ---')
print(f'PR-AUC:    {test_pr_auc:.4f}')
print(f'Precision: {test_metrics["precision"]:.4f}')
print(f'Recall:    {test_metrics["recall"]:.4f}')
print(f'F1 Score:  {test_metrics["f1"]:.4f}')
print(f'Total Expected Cost: ${test_metrics["expected_cost"]:,.2f}')
print(f'False Positives: {test_metrics["fp"]} (Cost: ${test_metrics["cost_fp"]:,.2f})')
print(f'False Negatives: {test_metrics["fn"]} (Cost: ${test_metrics["cost_fn"]:,.2f})')

## 9. Conclusion

- **Decision Optimization**: By minimizing the explicit financial cost (FP waste vs FN loss), we identified the most operationally efficient threshold.
- **Validation Safety**: Thresholds were tuned strictly on a validation split without touching the test set.
- **Model Selection**: The lowest Expected Monetary Cost was used to select the best predictive model for production deployment.
